# Name That LLM: offline submission (batch inference)

This notebook produces our competition submission. When you submit it, Kaggle **re-runs it
on ~4,000 hidden passages with the internet turned off**. While you edit it, it only sees 3
example test rows. The code must therefore work offline and must not assume a row count.

- **Code:** the project's package, attached as the dataset `ryqn00/name-that-llm-lib` and
  published automatically by CI on each version tag
  ([repo](https://github.com/Ryqn0/name-that-llm)).
- **Model, an A/B test:** with `USE_LM = True`, the TF-IDF champion `lr` (public LB 0.320)
  is blended with a second model that reads **Qwen 3 4B's hidden states**: the average of
  layers 6 to 9 over each passage's tokens. On fresh cross-validation folds the blend beat
  `lr` by +0.084 (95% CI [+0.066, +0.101]), see the
  [sweep notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-lm-layer-sweep). The
  text model is unchanged since the 0.320 submission, so the leaderboard difference
  measures what the LM features add. `USE_LM = False` gives back the text-only `lr`.
- **The LM** is attached as a Kaggle Model (`qwen-lm/qwen-3/transformers/4b-base`, Apache
  2.0) and runs on the T4 GPU in float16. torch and transformers come preinstalled in
  Kaggle's GPU image, so our package doesn't depend on them.

In [ ]:
# Internet is OFF, so install from the attached dataset. Search for the wheel instead of
# hard-coding the mount path: Kaggle's input layout has changed before (the competition data
# now sits under /kaggle/input/competitions/...).
import glob
import os

found = glob.glob("/kaggle/input/**/name_that_llm-*.whl", recursive=True)
wheel_dirs = {os.path.dirname(p) for p in found}
assert len(wheel_dirs) == 1, f"expected exactly one wheel folder, found: {wheel_dirs}"
WHEELS = wheel_dirs.pop()
print("installing from", WHEELS, "->", sorted(os.listdir(WHEELS)))

In [ ]:
# --no-index: never reach for PyPI (there's no internet). --find-links: take wheels from
# the attached folder. Packages Kaggle already has (pandas, scikit-learn, ...) stay as they are.
%pip install -q --no-index --find-links {WHEELS} name-that-llm

In [ ]:
import time

import numpy as np
import pandas as pd

import name_that_llm
from name_that_llm.config import BlendConfig, load_config
from name_that_llm.data import load_test, load_train
from name_that_llm.submit import predict_submission, predict_submission_blend

CHAMPION = "lr"  # the text-only champion: public LB 0.320
USE_LM = True  # P3b A/B test: blend lr with Qwen 3 hidden states. False = text-only lr
BLEND = "lm_blend"  # the BlendConfig chosen by the sweep notebook (05-lm-sweep)

blend_cfg = load_config(BLEND, model_cls=BlendConfig) if USE_LM else None
# The blend names its own text model: it was tuned together with that one.
text_cfg = load_config(blend_cfg.text_config if USE_LM else CHAMPION)
print("package", name_that_llm.__version__, "| text model:", text_cfg.name)
print("blend:", blend_cfg)

train, test = load_train(), load_test()  # validated: ids stay strings, labels are checked
print("train", train.shape, "| test", test.shape)

## LM features on the GPU

Each passage becomes one vector: the average, over its tokens, of the model's hidden states
in the blend's layers. We extract them for train **and** test, here, in the same run.

**Why recompute the training features** instead of attaching the ones the
[features notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-lm-features) saved? It
costs about 6 GPU-minutes, but it's one artifact less to keep in sync: train and test
features come from the same code, model and run, so they can't drift apart.

In [ ]:
extract_min = 0.0  # training-feature extraction time, added to the runtime check below
if USE_LM:
    import torch
    import transformers

    from name_that_llm.lm import find_model_dir, load_lm, pooled_hidden_states, select_layers

    print("torch", torch.__version__, "| transformers", transformers.__version__)
    # Fail in the first seconds, not after hours: the blend needs the GPU (accelerator: T4).
    assert torch.cuda.is_available(), "USE_LM = True needs a GPU"
    model, tokenizer = load_lm(find_model_dir(blend_cfg.lm_variation))  # searched, never hard-coded

    # Keep only the layers the blend uses (hidden_states index: 0 = embeddings, k = block k).
    # The forward pass computes every layer anyway; this just saves memory.
    LAYERS = range(blend_cfg.layer, blend_cfg.layer + blend_cfg.layer_window)

    def lm_features(texts):
        feats = pooled_hidden_states(
            texts,
            model,
            tokenizer,
            layers=LAYERS,
            batch_size=blend_cfg.batch_size,
            max_length=blend_cfg.max_length,
        )
        return select_layers(feats, 0, blend_cfg.layer_window)  # the mean of those layers

    start = time.time()
    lm_train = lm_features(train["text"].tolist())
    extract_min = (time.time() - start) / 60
    # STRICT on train: these are known passages, finite in the features notebook. A failure
    # here is a real problem, so stop before submitting anything.
    assert np.isfinite(lm_train).all(), "non-finite train features (see 04-lm-features)"

    lm_test = lm_features(test["text"].tolist())
    # LENIENT on test: float16 can overflow on an unusual passage, and one bad passage should
    # cost one prediction, not crash the 9-hour run. The TRAIN MEAN is the neutral input:
    # once scaled it's all zeros, so the LM head gives near-uniform probabilities and the
    # text model decides that passage.
    bad = ~np.isfinite(lm_test).all(axis=1)
    if bad.any():
        print(f"warning: {bad.sum()} test passages had non-finite LM features")
        lm_test[bad] = lm_train.mean(axis=0)
    print(f"LM features: train {lm_train.shape} in {extract_min:.1f} min, test {lm_test.shape}")

In [ ]:
start = time.time()
# Fits on ALL of train (CV was only for measuring), predicts test in order, and VALIDATES
# the result (every id exactly once, no nulls, only the 18 labels) before we write anything.
if USE_LM:
    sub = predict_submission_blend(text_cfg, blend_cfg, train, test, lm_train, lm_test)
else:
    sub = predict_submission(text_cfg, train, test)
sub.to_parquet("/kaggle/working/submission.parquet", index=False)
print(f"wrote submission.parquet: {len(sub)} rows in {time.time() - start:.0f}s")
sub.head()

## Runtime check

The hidden rerun has ~4,000 passages and a **9-hour** limit. We time the LM extraction and
a full fit plus prediction on 4,000 stand-in rows (training texts, resampled), then add the
training-feature extraction measured above. The total must be far below that limit.

In [ ]:
stand_in = pd.DataFrame(
    {
        "id": [f"t{i}" for i in range(4000)],
        "text": train["text"].sample(4000, replace=True, random_state=0).to_numpy(),
    }
)
start = time.time()
if USE_LM:
    lm_stand_in = lm_features(stand_in["text"].tolist())
    predict_submission_blend(text_cfg, blend_cfg, train, stand_in, lm_train, lm_stand_in)
else:
    predict_submission(text_cfg, train, stand_in)
minutes = extract_min + (time.time() - start) / 60
print(f"estimated hidden run: {minutes:.1f} min (limit: 540 min)")
assert minutes < 120, "far too slow for comfort; investigate before submitting"